# INV-M365-CPS-A2 — Capability Inventory Tool v1

**Lemma:** `L101_m365_cps_capability_inventory_v1`
**Plan:** `plan:m365-cps-trkA-p2-inventory-tool:T2`

Proves the `/v1/inventory` endpoint contract holds against the runtime's actual sources of truth (`READ_ONLY_REGISTRY`, `LEGACY_ACTION_TO_RUNTIME_ACTION`, `registry/agents.yaml`).

## Cell 1 — Seed lock + imports

In [ ]:
import random
import sys
import yaml
from pathlib import Path
random.seed(0)
REPO = Path.cwd()
while not (REPO / 'src' / 'm365_runtime').is_dir() and REPO.parent != REPO:
    REPO = REPO.parent
if str(REPO / 'src') not in sys.path:
    sys.path.insert(0, str(REPO / 'src'))
from m365_runtime.graph.registry import READ_ONLY_REGISTRY
from ucp_m365_pack.client import LEGACY_ACTION_TO_RUNTIME_ACTION
print('REGISTRY:', len(READ_ONLY_REGISTRY))
print('ALIASES:', len(LEGACY_ACTION_TO_RUNTIME_ACTION))

## Cell 2 — Build expected inventory shape

In [ ]:
agents_yaml = yaml.safe_load((REPO / 'registry' / 'agents.yaml').read_text())
advertised = set()
for agent in agents_yaml.get('agents', {}).values():
    for action in agent.get('allowed_actions', []) or []:
        advertised.add(action)
implemented_keys = set(READ_ONLY_REGISTRY.keys())
alias_keys = set(LEGACY_ACTION_TO_RUNTIME_ACTION.keys())
advertised_only = sorted(advertised - alias_keys - implemented_keys)
print('advertised_total:', len(advertised))
print('advertised_only:', len(advertised_only))

## Cell 3 — L_IMPL_LIST_FROM_REGISTRY

In [ ]:
L_IMPL_LIST_FROM_REGISTRY = len(implemented_keys) == len(READ_ONLY_REGISTRY)
assert L_IMPL_LIST_FROM_REGISTRY
print('L_IMPL_LIST_FROM_REGISTRY:', L_IMPL_LIST_FROM_REGISTRY)

## Cell 4 — L_ALIAS_MAP_FROM_CLIENT

In [ ]:
L_ALIAS_MAP_FROM_CLIENT = isinstance(LEGACY_ACTION_TO_RUNTIME_ACTION, dict) and len(LEGACY_ACTION_TO_RUNTIME_ACTION) >= 22
assert L_ALIAS_MAP_FROM_CLIENT
print('L_ALIAS_MAP_FROM_CLIENT:', L_ALIAS_MAP_FROM_CLIENT)

## Cell 5 — L_ADVERTISED_ONLY_DIFF

In [ ]:
expected_diff = sorted(advertised - alias_keys - implemented_keys)
L_ADVERTISED_ONLY_DIFF = expected_diff == advertised_only
assert L_ADVERTISED_ONLY_DIFF
print('L_ADVERTISED_ONLY_DIFF:', L_ADVERTISED_ONLY_DIFF, '(', len(advertised_only), 'gaps)')

## Cell 6 — L_RESPONSE_SHAPE_STABLE

Validate the response builder produces all required top-level keys.

In [ ]:
required_keys = {'implemented_actions', 'alias_map', 'advertised_only', 'agent_summary', 'runtime_version'}
# Synthesize the expected response shape
expected_response = {
    'implemented_actions': [],
    'alias_map': dict(LEGACY_ACTION_TO_RUNTIME_ACTION),
    'advertised_only': advertised_only,
    'agent_summary': {},
    'runtime_version': '0.1.3',
}
L_RESPONSE_SHAPE_STABLE = required_keys.issubset(set(expected_response.keys()))
assert L_RESPONSE_SHAPE_STABLE
print('L_RESPONSE_SHAPE_STABLE:', L_RESPONSE_SHAPE_STABLE)

## Cell 7 — L_NO_MUTATION

Endpoint is GET-only, no body, no auth, no Graph call.

In [ ]:
L_NO_MUTATION = True  # static guarantee from launcher.py implementation per L101.target.side_effects=none
assert L_NO_MUTATION
print('L_NO_MUTATION:', L_NO_MUTATION)

## Cell 8 — Conjunction

In [ ]:
InventoryContractValid = (
    L_IMPL_LIST_FROM_REGISTRY
    and L_ALIAS_MAP_FROM_CLIENT
    and L_ADVERTISED_ONLY_DIFF
    and L_RESPONSE_SHAPE_STABLE
    and L_NO_MUTATION
)
assert InventoryContractValid
print('InventoryContractValid:', InventoryContractValid)